<a href="https://colab.research.google.com/github/JCF-the-creator/Multi_Motorsport_Analytics_Desempenho_e_Estrategia/blob/main/Resultados_Rally_2025_Completo%20-%20Copia/ConverterJsonEmCsvRally.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import json
import pandas as pd
from pathlib import Path
import os
import pandas as pd
import numpy as np

#de update em todos os arquivos json a serem convertidos no caminho "/content"

def json_rally_to_csv(pasta_entrada: str = ".", pasta_saida: str = "."):
    """
    Converte todos os arquivos JSON de resultados de rally em CSVs.

    - Lê todos os *.json da pasta de entrada
    - Injeta COMPETICAO e INFO_GERAL (do CABECALHO) em cada linha de DADOS
    - Salva o CSV com o mesmo nome do JSON na pasta de saída
    """
    pasta_entrada = Path(pasta_entrada)
    pasta_saida = Path(pasta_saida)
    pasta_saida.mkdir(parents=True, exist_ok=True)

    arquivos_json = list(pasta_entrada.glob("*.json"))

    if not arquivos_json:
        print(f"Nenhum arquivo .json encontrado em: {pasta_entrada.resolve()}")
        return

    print(f"{len(arquivos_json)} arquivo(s) encontrado(s).\n")

    for caminho_json in arquivos_json:
        try:
            with open(caminho_json, "r", encoding="utf-8") as f:
                dados = json.load(f)

            # Extrai campos do cabeçalho
            cabecalho = dados.get("CABECALHO", {})
            competicao = cabecalho.get("COMPETICAO", "")
            info_geral = cabecalho.get("INFO_GERAL", "")

            registros = dados.get("DADOS", [])
            if not registros:
                print(f"[AVISO] {caminho_json.name}: sem registros em DADOS. Pulando.")
                continue

            df = pd.DataFrame(registros)

            # Insere as colunas do cabeçalho nas duas primeiras posições
            df.insert(0, "COMPETICAO", competicao)
            df.insert(1, "INFO_GERAL", info_geral)

            nome_csv = caminho_json.stem + ".csv"
            caminho_csv = pasta_saida / nome_csv

            df.to_csv(caminho_csv, index=False, encoding="utf-8-sig")
            print(f"[OK] {caminho_json.name}  →  {nome_csv}  ({len(df)} linhas)")

        except Exception as e:
            print(f"[ERRO] {caminho_json.name}: {e}")

    print("\nConcluído.")


if __name__ == "__main__":
    import sys

    # Uso: python json_to_csv_rally.py [pasta_entrada] [pasta_saida]
    # Se não informar argumentos, usa a pasta atual para entrada e saída.
    entrada = Path("/content/")
    saida = Path("/content/csv")
    saida.mkdir(parents=True, exist_ok=True)

    csv_corrigido = Path("/content/csv_corrigido")
    csv_corrigido.mkdir(parents=True, exist_ok=True)

    json_rally_to_csv(pasta_entrada=entrada, pasta_saida=saida)

    arquivos_csv = list(saida.glob("*.csv"))

    for caminho_csv in arquivos_csv:
      # 1. Carregar o arquivo CSV
      # Substitua pelo caminho correto do seu arquivo se necessário
      df = pd.read_csv(caminho_csv)

      # 6. Dividir "TOTAL DIF.LIDER DIF.ANTERIOR" em 3 colunas
      # Primeiro tratamos possíveis valores nulos ou esquisitos nessa coluna original

      try:
        # 2. Renomear a coluna (POS)CAT para CAT
        df = df.rename(columns={'(POS)CAT': 'CAT'})

        # 3. Deletar o texto entre parênteses da coluna CAT
        df['CAT'] = df['CAT'].astype(str).str.replace(r'\(.*\)', '', regex=True).str.strip()

        # 4. Deletar o texto entre parênteses das colunas de dias (DIA 1, DIA 2, DIA 3)
        # Fazemos um loop para aplicar em todas as colunas que começam com "DIA"
        colunas_dias = [col for col in df.columns if col.startswith('DIA ')]
        for col in colunas_dias:
            df[col] = df[col].astype(str).str.replace(r'\(.*\)', '', regex=True).str.strip()
      except:
        print(f"Erro ao dividir coluna CAT do arquivo {caminho_csv}")
        pass

      try:
        #tem arquivos com CATEGORIA (POS)
        df = df.rename(columns={'CATEGORIA (POS)': 'CAT'})

        # 3. Deletar o texto entre parênteses da coluna CAT
        df['CAT'] = df['CAT'].astype(str).str.replace(r'\(.*\)', '', regex=True).str.strip()

        # 4. Deletar o texto entre parênteses das colunas de dias (DIA 1, DIA 2, DIA 3)
        # Fazemos um loop para aplicar em todas as colunas que começam com "DIA"
        colunas_dias = [col for col in df.columns if col.startswith('DIA ')]
        for col in colunas_dias:
            df[col] = df[col].astype(str).str.replace(r'\(.*\)', '', regex=True).str.strip()
      except:
        print(f"Erro ao dividir coluna CAT do arquivo {caminho_csv}")
        pass

      try:
        # 5. Dividir a coluna PENAL BONUS em duas (PENAL e BONUS)
        # Substitui '* * *' por '00:00:00' (ou 0) para manter o padrão antes da divisão
        df['PENAL BONUS'] = df['PENAL BONUS'].astype(str).str.replace('* * *', '00:00:00')

        # Divide a coluna com base no espaço
        divisao_penal_bonus = df['PENAL BONUS'].str.split(' ', expand=True)

        # Garante que temos duas colunas preenchidas (caso falte o bônus em alguma linha)
        df['PENAL'] = divisao_penal_bonus[0]
        df['BONUS'] = divisao_penal_bonus[1] if divisao_penal_bonus.shape[1] > 1 else '00:00:00'

        # Se o bônus ficou nulo ou vazio, vira '0'
        df['BONUS'] = df['BONUS'].fillna('00:00:00').replace('', '00:00:00')

      # Remove a coluna original antiga
        df = df.drop(columns=['PENAL BONUS'])
      except:
        print(f"Erro ao dividir coluna PENAL BONUS do arquivo {caminho_csv}")
        pass

      try:
        # tem arquivos com acento na coluna
        df['PENAL BONUS'] = df['PENAL BÔNUS'].astype(str).str.replace('* * *', '00:00:00')

        # Divide a coluna com base no espaço
        divisao_penal_bonus = df['PENAL BONUS'].str.split(' ', expand=True)

        # Garante que temos duas colunas preenchidas (caso falte o bônus em alguma linha)
        df['PENAL'] = divisao_penal_bonus[0]
        df['BONUS'] = divisao_penal_bonus[1] if divisao_penal_bonus.shape[1] > 1 else '00:00:00'

        # Se o bônus ficou nulo ou vazio, vira '0'
        df['BONUS'] = df['BONUS'].fillna('00:00:00').replace('', '00:00:00')

        # Remove a coluna original antiga
        df = df.drop(columns=['PENAL BÔNUS'])
        df = df.drop(columns=['PENAL BONUS'])

      except:
        pass

      df['TOTAL DIF.LIDER DIF.ANTERIOR'] = df['TOTAL DIF.LIDER DIF.ANTERIOR'].astype(str).str.strip()

      # Divide com base nos espaços em branco
      divisao_totais = df['TOTAL DIF.LIDER DIF.ANTERIOR'].str.split(' ', expand=True)

      df['TOTAL'] = divisao_totais[0]
      df['DIF.LIDER'] = divisao_totais[1] if divisao_totais.shape[1] > 1 else '0'
      df['DIF.ANTERIOR'] = divisao_totais[2] if divisao_totais.shape[1] > 2 else '0'

      # Remove a coluna original antiga
      df = df.drop(columns=['TOTAL DIF.LIDER DIF.ANTERIOR'])

      # 7. Tratar valores nulos (NaN) e resquícios de texto 'nan' para se tornarem 0
      df = df.fillna(0)
      df = df.replace(['nan', 'NaN', 'None', ''], 0)

      caminho_csv = str(caminho_csv).removeprefix("/content/csv/")

      # Salvar o novo arquivo limpo
      df.to_csv(f"/content/csv_corrigido/{caminho_csv}", index=False)

    os.listdir()

    !zip Rally_Csv_corrigido.zip \
    csv_corrigido/*.csv


22 arquivo(s) encontrado(s).

[OK] Resultado-Prova_2-RioNegrinho25-Oficial_corrigido.json  →  Resultado-Prova_2-RioNegrinho25-Oficial_corrigido.csv  (15 linhas)
[OK] Result-D4-Acumulado-Motos_Geral-RN25-v1_corrigido.json  →  Result-D4-Acumulado-Motos_Geral-RN25-v1_corrigido.csv  (12 linhas)
[OK] Resultado-Prova_1-RioNegrinho25-Oficial_1_corrigido.json  →  Resultado-Prova_1-RioNegrinho25-Oficial_1_corrigido.csv  (14 linhas)
[OK] Result-Dia_3-Acumulado-Geral_UTV-MB25-Of-v1_corrigido.json  →  Result-Dia_3-Acumulado-Geral_UTV-MB25-Of-v1_corrigido.csv  (58 linhas)
[OK] Result-Acumulado_ate_D6-Geral_UTV-Sertoes25-ExOf-v2_corrigido.json  →  Result-Acumulado_ate_D6-Geral_UTV-Sertoes25-ExOf-v2_corrigido.csv  (61 linhas)
[OK] Result-Dia_1-Geral_Motos-Serra_Azul-Of-v1_corrigido.json  →  Result-Dia_1-Geral_Motos-Serra_Azul-Of-v1_corrigido.csv  (22 linhas)
[OK] Result-Acumulado-Geral_Motos-Rota_Sudeste25-Of-v1_corrigido.json  →  Result-Acumulado-Geral_Motos-Rota_Sudeste25-Of-v1_corrigido.csv  (19 l